In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Load the dataset
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

# 2. Data Preparation
categorical = list(df.select_dtypes(include=['object']).columns)
numerical = list(df.select_dtypes(exclude=['object']).columns)

if 'converted' in numerical:
    numerical.remove('converted')
if 'converted' in categorical:
    categorical.remove('converted')

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

# Question 1
print("--- Question 1 ---")
print(f"Most frequent industry: {df['industry'].mode()[0]}\n")

# Question 2
print("--- Question 2 ---")
corr_matrix = df[numerical].corr()
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]
for f1, f2 in pairs:
    print(f"Correlation between {f1} and {f2}: {corr_matrix.loc[f1, f2]:.4f}")
print()

# Split the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

# Question 3
print("--- Question 3 ---")
def mutual_info_converted(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted)
print(mi.round(2).sort_values(ascending=False))
print()

# Question 4
print("--- Question 4 ---")
dv = DictVectorizer(sparse=False)
train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
baseline_acc = accuracy_score(y_val, y_pred)
print(f"Accuracy: {round(baseline_acc, 2)}\n")

# Question 5
print("--- Question 5 ---")
features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feature in features_to_check:
    subset_features = categorical + numerical
    subset_features.remove(feature)
    
    train_dict_sub = df_train[subset_features].to_dict(orient='records')
    val_dict_sub = df_val[subset_features].to_dict(orient='records')
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(train_dict_sub)
    X_val_sub = dv_sub.transform(val_dict_sub)
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    y_pred_sub = model_sub.predict(X_val_sub)
    acc_sub = accuracy_score(y_val, y_pred_sub)
    diff = baseline_acc - acc_sub
    print(f"Feature: {feature} | Difference: {diff:.5f}")
print()

# Question 6
print("--- Question 6 ---")
for c_val in [0.000001, 0.00001, 0.0001, 0.001]:
    model_reg = LogisticRegression(solver='liblinear', C=c_val, max_iter=1000, random_state=42)
    model_reg.fit(X_train, y_train)
    
    y_pred_reg = model_reg.predict(X_val)
    acc_reg = accuracy_score(y_val, y_pred_reg)
    print(f"C={c_val} | Accuracy={round(acc_reg, 3)}")

/tmp/ipykernel_45893/3297587229.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = list(df.select_dtypes(include=['object']).columns)


--- Question 1 ---
Most frequent industry: technology

--- Question 2 ---
Correlation between interaction_count and lead_score: 0.9157
Correlation between number_of_courses_viewed and lead_score: 0.7572
Correlation between number_of_courses_viewed and interaction_count: 0.7216
Correlation between annual_income and interaction_count: 0.1228

--- Question 3 ---
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

--- Question 4 ---
Accuracy: 0.65

--- Question 5 ---
Feature: lead_source | Difference: 0.00300
Feature: number_of_courses_viewed | Difference: 0.00200
Feature: interaction_count | Difference: 0.04400

--- Question 6 ---
C=1e-06 | Accuracy=0.598
C=1e-05 | Accuracy=0.598
C=0.0001 | Accuracy=0.613
C=0.001 | Accuracy=0.645
